# Simple CLI chat loop that sends user queries to the GIS agent and prints the results


In [4]:
from core.planner import process_user_input
from core.sql_builder import plan_to_sql, enrich_plan
from mcp_c_s.mcp_client import query_postgis 
from ui.map_utils import showMap

In [6]:
# async def chat_loop():
#     print("GIS agent chat – type 'quit' to stop.\n")

#     while True:
#         user_q = input("You: ").strip()
#         print("'''''''''''''''''''''''''''''''''''''''''''")

#         if user_q.lower() in ("quit", "exit", "q"):
#             print("Welcome back!")
#             break

#         plan = process_user_input(user_q)
#         if not isinstance(plan, dict):
#             continue

#         sql = plan_to_sql(plan)

#         df = await query_postgis(sql)  

#         if df is None or df.empty:
#             print("No results found.")
#             continue

#         display(df.head(10))
#         display(showMap(df))
# await chat_loop()


In [ ]:
# async def chat_once(user_q):
#     plan = process_user_input(user_q)
#     #plan = enrich_plan(plan)
#     if not isinstance(plan, dict):
#         print(plan)
#         return
#     plan = enrich_plan(plan) 
    
#     sql = plan_to_sql(plan)
#     df = await query_postgis(sql)

#     if df is None or df.empty:
#         print("No results.")
#         return

#     display(showMap(df))


In [ ]:
# tools = [
#   {"type":"function","function":{
#     "name":"make_plan",
#     "parameters":{"type":"object","properties":{"user_text":{"type":"string"}}, "required":["user_text"]}
#   }},
#   {"type":"function","function":{
#     "name":"plan_to_sql",
#     "parameters":{"type":"object","properties":{"plan_json":{"type":"string"}}, "required":["plan_json"]}
#   }},
#   {"type":"function","function":{
#     "name":"run_postgis_query",
#     "parameters":{"type":"object","properties":{"sql":{"type":"string"}}, "required":["sql"]}
#   }},
# ]


In [ ]:
# import json
# import pandas as pd
# from ui.map_utils import showMap

# async def agent(user_text: str):
#     messages = [{"role":"user","content":user_text}]
#     last_df = None

#     while True:
#         resp = client.chat.completions.create(
#             model=MODEL,
#             messages=messages,
#             tools=tools,
#             tool_choice="auto",
#         )

#         msg = resp.choices[0].message
#         messages.append(msg)

#         if not msg.tool_calls:
#             return msg.content, last_df

#         for tc in msg.tool_calls:
#             name = tc.function.name
#             args = json.loads(tc.function.arguments)

#             if name == "make_plan":
#                 out = await mcp_call("make_plan", args)

#             elif name == "plan_to_sql":
#                 out = await mcp_call("plan_to_sql_tool", args)

#             elif name == "run_postgis_query":
#                 out = await mcp_call("run_postgis_query", args)

#                 # MCP tool returns JSON rows -> DataFrame
#                 try:
#                     rows = json.loads(out) if out else []
#                     last_df = pd.DataFrame(rows)
#                 except Exception:
#                     last_df = pd.DataFrame()

#             else:
#                 out = ""

#             messages.append({
#                 "role": "tool",
#                 "tool_call_id": tc.id,
#                 "content": out
#             })


In [ ]:
text, df = await agent("5 buildings in drammen")
df.head()
showMap(df)


'pong'

In [2]:
from mcp_c_s.mcp_client import mcp_call
await mcp_call("ping", {})


'pong'

In [ ]:
# import json
# import pandas as pd
# from mcp.client.session import ClientSession
# from mcp.client.stdio import stdio_client
# from mcp_c_s.mcp_client import _server_params

# async def agent(user_text: str):
#     messages = [{"role":"user","content":user_text}]
#     last_df = None
#     server_params = _server_params()

#     async with stdio_client(server_params) as (read, write):
#         async with ClientSession(read, write) as session:
#             await session.initialize()

#             while True:
#                 resp = client.chat.completions.create(
#                     model=MODEL,
#                     messages=messages,
#                     tools=tools,
#                     tool_choice="auto",
#                 )

#                 msg = resp.choices[0].message
#                 messages.append(msg)

#                 if not msg.tool_calls:
#                     return msg.content, last_df

#                 for tc in msg.tool_calls:
#                     name = tc.function.name
#                     args = json.loads(tc.function.arguments)

#                     if name == "make_plan":
#                         res = await session.call_tool("make_plan", arguments=args)
#                         out = res.content[0].text if res.content else ""

#                     elif name == "plan_to_sql":
#                         res = await session.call_tool("plan_to_sql_tool", arguments=args)
#                         out = res.content[0].text if res.content else ""

#                     elif name == "run_postgis_query":
#                         res = await session.call_tool("run_postgis_query", arguments=args)
#                         out = res.content[0].text if res.content else ""
#                         try:
#                             last_df = pd.DataFrame(json.loads(out)) if out else pd.DataFrame()
#                         except Exception:
#                             last_df = pd.DataFrame()
#                     else:
#                         out = ""

#                     messages.append({
#                         "role": "tool",
#                         "tool_call_id": tc.id,
#                         "content": out
#                     })


In [ ]:
text, df = await agent("5 buildings in drammen")
df.head()
showMap(df)

In [1]:
import json
import pandas as pd
from mcp_c_s.mcp_client import mcp_call
from ui.map_utils import showMap

plan_json = await mcp_call("make_plan", {"user_text": "5 buildings in drammen"})
sql = await mcp_call("plan_to_sql_tool", {"plan_json": plan_json})
rows_json = await mcp_call("run_postgis_query", {"sql": sql})

df = pd.DataFrame(json.loads(rows_json)) if rows_json else pd.DataFrame()
df.head(), showMap(df)


(      gid     osm_id  code    fclass  name    type  \
 0  930360  760677244  1500  building  None    None   
 1  930370  760677254  1500  building  None    None   
 2  930374  760677258  1500  building  None   house   
 3  930377  760677261  1500  building  None  garage   
 4  930387  760677271  1500  building  None  garage   
 
                                                 geom  \
 0  0106000020E96400000100000001030000000100000005...   
 1  0106000020E96400000100000001030000000100000005...   
 2  0106000020E96400000100000001030000000100000009...   
 3  0106000020E96400000100000001030000000100000005...   
 4  0106000020E96400000100000001030000000100000005...   
 
                                             wkt_geom  
 0  MULTIPOLYGON(((10.389411300000003 59.6247735,1...  
 1  MULTIPOLYGON(((10.390721300000001 59.6233296,1...  
 2  MULTIPOLYGON(((10.3905905 59.623847000000005,1...  
 3  MULTIPOLYGON(((10.3902385 59.624382700000005,1...  
 4  MULTIPOLYGON(((10.3922969 59.6236909,10.

In [10]:
df.columns


RangeIndex(start=0, stop=0, step=1)

In [ ]:
import json
import pandas as pd
from core.llm import LLMClient
from mcp_c_s.mcp_client import mcp_call

llm = LLMClient()
llm_client = llm.client
MODEL = llm.model

SYSTEM_TOOL_POLICY = """
You are a GIS agent. You have ONE tool: gis_query.
When the user asks a GIS question, you MUST call gis_query with their query.
After you receive the results, summarize them briefly.
"""

tools = [
  {"type":"function","function":{
    "name":"gis_query",
    "parameters":{"type":"object","properties":{
      "user_query":{"type":"string"},
      "k":{"type":"integer","default":8}
    }, "required":["user_query"]}
  }},
]

async def agent_tools(user_text: str, max_turns: int = 3):
    messages = [
        {"role": "system", "content": SYSTEM_TOOL_POLICY},
        {"role": "user", "content": user_text},
    ]

    last_df = None
    have_results = False

    for turn in range(max_turns):
        print(f"Turn {turn + 1}/{max_turns}")

        resp = llm.client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto",
            temperature=0,
            max_tokens=500,
        )

        msg = resp.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            if have_results:
                return msg.content, last_df
            messages.append({
                "role": "system",
                "content": "You MUST call gis_query before answering."
            })
            continue

        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)

            print(f"Calling gis_query: {args.get('user_query')}")

            out = await mcp_call("gis_query", args)

            response = json.loads(out) if out else {}           
            rows = response.get("data", {}).get("rows", []) if response.get("ok") else []

            last_df = pd.DataFrame(rows)
            have_results = True

            print(f"Got {len(rows)} rows")

            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps({"row_count": len(rows)})
            })

    return "Stopped: max_turns reached.", last_df


async def chat_agent(user_q: str):
    text, df = await agent_tools(user_q)
    print(f"Agent: {text}")
    print(f"Rows: {len(df)}")
    return df


In [2]:
from ui.map_utils import showMap



df = await chat_agent("5 buildings in drammen")
m = showMap(df)
m


Turn 1/3
Calling gis_query: 5 buildings in Drammen
Got 5 rows
Turn 2/3
Agent: I found information on 5 buildings in Drammen. If you need more specific details about these buildings, please let me know!
Rows: 5


In [ ]:
from core.llm import LLMClient
llm = LLMClient()
client = llm.client
MODEL = llm.model


In [8]:
df = await chat_agent("Find buildings within 50 meters of fotrute_senterlinje and within 50 meters of sykkelrute_senterlinje ")
m = showMap(df)
m


Turn 1/3
Calling gis_query: Find buildings within 50 meters of fotrute_senterlinje and within 50 meters of sykkelrute_senterlinje
Got 200 rows
Turn 2/3

Agent: The query returned 200 buildings located within 50 meters of both the "fotrute_senterlinje" and "sykkelrute_senterlinje." The results include various attributes such as their IDs, classification codes, names, types, and geometrical data.
DataFrame: 200 rows


In [3]:
df = await chat_agent("Find 100 buildings within 50 meters of fotrute_senterlinje and within 50 meters of sykkelrute_senterlinje in oslo ")
m = showMap(df)
m


Turn 1/3
Calling gis_query: Find 100 buildings within 50 meters of fotrute_senterlinje and within 50 meters of sykkelrute_senterlinje in Oslo
Got 100 rows
Turn 2/3

Agent: The query returned 100 buildings located within 50 meters of both the "fotrute_senterlinje" and "sykkelrute_senterlinje" in Oslo. The results include various attributes such as IDs, classifications, names, and geometrical data for each building.
DataFrame: 100 rows


In [5]:
prompt =     "Find 100 buildings within 50 meters of fotrute_senterlinje and within 50 meters of sykkelrute_senterlinje in Oslo"
plan = process_user_input(prompt)
print(json.dumps(plan, indent=2))


{
  "operation": "select_buffer",
  "layer": "public.buildings",
  "target_layer": "public.sykkelrute_senterlinje",
  "buffer_meters": 50,
  "limit": 100,
  "where_clause": "(ST_Intersects(\n  a.geom,\n  ST_Transform(ST_GeomFromGeoJSON('{\"type\": \"MultiPolygon\", \"coordinates\": [[[[10.772540087618275, 59.823683530026045], [10.77255201092786, 59.8236852624025], [10.772748507653638, 59.82372557287424], [10.772753884633197, 59.82372682486515], [10.773080490054754, 59.82379164186793], [10.773227971993084, 59.823808704925455], [10.773355049742417, 59.8238641350873], [10.773440299629806, 59.82387766738364], [10.773516551507655, 59.82392130186637], [10.77361620528945, 59.82395642769149], [10.773732916860714, 59.824002082813465], [10.773824577427032, 59.824053765694266], [10.773911535588486, 59.824134070441325], [10.774050088780955, 59.82419337570815], [10.774036873585802, 59.82420463977084], [10.774066929115081, 59.82422251069577], [10.774057663395238, 59.82428569351676], [10.774118011636

In [6]:
generated_sql = plan_to_sql(plan)
print(generated_sql)


SELECT a.*, ST_AsText(ST_Transform(a.geom, 4326)) AS wkt_geom
    FROM public.buildings a
    JOIN public.sykkelrute_senterlinje b
      ON a.geom && ST_Expand(b.geom, 50)
     AND ST_DWithin(a.geom, b.geom, 50)
    WHERE (ST_Intersects(
  a.geom,
  ST_Transform(ST_GeomFromGeoJSON('{"type": "MultiPolygon", "coordinates": [[[[10.772540087618275, 59.823683530026045], [10.77255201092786, 59.8236852624025], [10.772748507653638, 59.82372557287424], [10.772753884633197, 59.82372682486515], [10.773080490054754, 59.82379164186793], [10.773227971993084, 59.823808704925455], [10.773355049742417, 59.8238641350873], [10.773440299629806, 59.82387766738364], [10.773516551507655, 59.82392130186637], [10.77361620528945, 59.82395642769149], [10.773732916860714, 59.824002082813465], [10.773824577427032, 59.824053765694266], [10.773911535588486, 59.824134070441325], [10.774050088780955, 59.82419337570815], [10.774036873585802, 59.82420463977084], [10.774066929115081, 59.82422251069577], [10.7740576633952

In [10]:



print("##############################################JSON plan ############################################################# \n \n")

prompt = "5 buildings in drammen"
plan = process_user_input(prompt)
print(json.dumps(plan, indent=2))


print("##############################################Generated SQL ############################################################# \n \n")
generated_sql = plan_to_sql(plan)
print(generated_sql)

##############################################JSON plan ############################################################# 
 

{
  "operation": "select_by_attribute",
  "layer": "public.buildings",
  "target_layer": null,
  "buffer_meters": null,
  "limit": 5,
  "where_clause": "(ST_Intersects(\n  a.geom,\n  ST_Transform(ST_GeomFromGeoJSON('{\"type\": \"MultiPolygon\", \"coordinates\": [[[[10.365327087222594, 59.54397783982753], [10.365044810512677, 59.54407764306574], [10.36457371245938, 59.5443124939393], [10.364153754899963, 59.54451165237559], [10.363845946209377, 59.54467332677734], [10.363534377550932, 59.54518006575781], [10.36420807789067, 59.54702377985666], [10.364702007562482, 59.547123965021456], [10.365597952940346, 59.54767427163296], [10.36551079381592, 59.54772316596036], [10.365449906721947, 59.54779098078841], [10.365423247014924, 59.547835365795926], [10.365404334334306, 59.54790583907392], [10.3653830393578, 59.54795886898143], [10.365363130932916, 59.548008260450516], [

In [ ]:
df = await chat_agent(
    "Show me buildings larger than 500 square meters that are located within 200 meters of flood zones in Kristiansand"
)
m = showMap(df)
m


Turn 1/3
Calling gis_query: buildings larger than 500 square meters within 200 meters of flood zones in Kristiansand


In [5]:
await chat_once("Find buildings within 50 meters of fotrute_senterlinje and within 50 meters of sykkelrute_senterlinje ")


In [2]:
from ui.map_utils import showMap



df = await chat_agent("Find flomsoner that overlap arealbruk_skogbonitet")
m = showMap(df)
m

Turn 1/3
Calling gis_query: Find flomsoner that overlap arealbruk_skogbonitet
Got 200 rows
Turn 2/3

Agent: The query found 200 records of "flomsoner" (flood zones) that overlap with "arealbruk_skogbonitet" (land use forest productivity). The results include various attributes such as IDs, types, accuracy, and geometrical data. If you need more specific details or further analysis, please let me know!
DataFrame: 200 rows


In [7]:
await chat_once("Find flomsoner that overlap arealbruk_skogbonitet")


In [4]:
await chat_once("Find 1000 buildings that intersect flood zones")


In [5]:
await chat_once("Find 50 buildings that intersect flood zones in Oslo")


In [5]:
await chat_once("10000 buldings in Oslo")


In [3]:
await chat_once("50 buildings with area larger than 200 square meters")


In [3]:
await chat_once("100 buildings within 100 meters of skiloype_senterlinje")

In [11]:
await chat_once("200 buildings within 50 meters of fotrute_senterlinje in bergen")

In [ ]:
await chat_once("buildings where ST_Area(geom) > 200")

In [6]:
await chat_once("200 buildings within 75 meters of sykkelrute_senterlinje")

In [7]:
await chat_once("200 buildings intersect flomsoner")

In [8]:
await chat_once("200 buildings intersect flomsoner in Oslo")

In [ ]:
await chat_once("100 nearest buildings to skiloype_senterlinje")

In [ ]:
await chat_once("show 200 buildings larger than 200 square meters")


In [8]:
await chat_once("15 buldings in drammen")
